# 🚀 Fine-tune NLLB-Pruned-370M for Novel Translation (ZH-VI, VI-ZH, EN-VI, JA-VI, KO-VI)

**Mục tiêu:**
- Fine-tune model **NLLB-200 Pruned 370M** (đã cắt tỉa từ vựng từ 256k xuống 65k tokens) bằng phương pháp **PEFT LoRA**.
- Tập trung tối ưu chất lượng dịch văn phong truyện tiên hiệp/kiếm hiệp/đô thị (Chinese $\to$ Vietnamese 87%), đồng thời bảo toàn khả năng dịch hai chiều và các cặp ngôn ngữ khác (VI-ZH 7%, EN-VI, JA-VI, KO-VI).
- Sau khi huấn luyện, merge LoRA trọng số trở lại base model để sẵn sàng convert sang **ONNX INT8** tích hợp vào **DrDucBook**.

## 1. Kiểm tra GPU & Cài đặt thư viện cần thiết

In [ ]:
!nvidia-smi

!pip install -q -U transformers datasets peft accelerate sentencepiece sacrebleu

## 2. Tải hoặc Mount dữ liệu huấn luyện
Bạn có thể chọn 1 trong 2 cách:
- **Cách 1 (Google Drive):** Mount Drive nếu bạn đã upload file `nllb_novel_training_bundle.zip` lên Drive.
- **Cách 2 (Upload trực tiếp):** Upload file `nllb_novel_training_bundle.zip` vào thư mục hiện tại của Colab.

In [ ]:
# Tùy chọn 1: Mount Google Drive (nếu lưu trên Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/nllb_novel_training_bundle.zip /content/

# Giải nén bundle (chứa nllb_pruned_370m, dataset_novel, testdata)
import os
if os.path.exists("nllb_novel_training_bundle.zip"):
    !unzip -q nllb_novel_training_bundle.zip -d /content/
    print("Đã giải nén bundle thành công!")
else:
    print("Vui lòng upload nllb_novel_training_bundle.zip vào /content/ hoặc mount Google Drive.")

## 3. Khởi tạo Tokenizer & Base Model Pruned (419M Params, 65k Vocab)

In [ ]:
import torch
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "/content/nllb_pruned_370m"

print(f"Loading pruned tokenizer from {MODEL_PATH}...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_PATH)
print(f"Vocab size: {len(tokenizer.get_vocab())}")

print(f"Loading base model in FP16...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

## 4. Cấu hình PEFT LoRA (Low-Rank Adaptation)
- `r = 16`, `lora_alpha = 32`, `lora_dropout = 0.05`
- `target_modules = ["q_proj", "v_proj", "k_proj", "out_proj"]`
- Chỉ cập nhật ~1.5% tổng số tham số, giúp huấn luyện nhanh và tiết kiệm VRAM đáng kể.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Tải và Tokenize Tập dữ liệu Huấn luyện (465,000 cặp câu)

In [ ]:
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq

TRAIN_FILE = "/content/dataset_novel/train.jsonl"
EVAL_FILE = "/content/dataset_novel/eval.jsonl"

print("Loading dataset JSONL...")
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": EVAL_FILE})
print(dataset)

MAX_LENGTH = 256

def preprocess_batch(batch):
    input_ids_list = []
    labels_list = []
    for src, tgt, src_lang, tgt_lang in zip(
        batch["source"], batch["target"], batch["src_lang"], batch["tgt_lang"]
    ):
        tokenizer.src_lang = src_lang
        tokenizer.tgt_lang = tgt_lang
        inp = tokenizer(src, max_length=MAX_LENGTH, truncation=True)
        out = tokenizer(text_target=tgt, max_length=MAX_LENGTH, truncation=True)
        input_ids_list.append(inp["input_ids"])
        labels_list.append(out["input_ids"])
    return {"input_ids": input_ids_list, "labels": labels_list}

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=2000,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, pad_to_multiple_of=8)
print("Tokenization finished successfully!")

## 6. Huấn luyện với Seq2SeqTrainer

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

OUTPUT_DIR = "/content/nllb_lora_checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,          # 16 on T4 / 32 on A100
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,          # Effective batch size = 32
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    num_train_epochs=2,                      # 2 epochs ~ 28k steps
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    fp16=True,
    predict_with_generate=False,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()

## 7. Đánh giá chất lượng với Golden Test Set (Văn phong Tiên Hiệp & Đô Thị)

In [ ]:
import json

GOLDEN_FILE = "/content/testdata/golden_test_set_literary.json"

if os.path.exists(GOLDEN_FILE):
    with open(GOLDEN_FILE, "r", encoding="utf-8") as f:
        gold_data = json.load(f)
    
    model.eval()
    tokenizer.src_lang = "zho_Hans"
    target_lang_id = tokenizer.convert_tokens_to_ids("vie_Latn")
    
    print("=" * 80)
    print("KẾT QUẢ DỊCH THỰC TẾ TRÊN TẬP GOLDEN TEST SET")
    print("=" * 80)
    
    for i, tc in enumerate(gold_data.get("test_cases", [])[:10], 1):
        zh = tc["zh"]
        expected = tc.get("expected_vi", "")
        inputs = tokenizer(zh, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=target_lang_id,
                max_new_tokens=256,
                num_beams=4
            )
        pred = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
        print(f"[{i:02d}] ZH  : {zh}")
        print(f"     DỊCH: {pred}")
        print(f"     CHUẨN: {expected}\n")

## 8. Merge LoRA Trọng số vào Base Model & Xuất ra Thư mục Chuẩn ONNX

In [ ]:
EXPORT_DIR = "/content/nllb_pruned_novel_finetuned"
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Merging LoRA adapter back into base weights -> {EXPORT_DIR}...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

print("Nén thư mục model đã fine-tune thành zip để tải về:")
!zip -r /content/nllb_pruned_novel_finetuned.zip /content/nllb_pruned_novel_finetuned
print("Hoàn tất! Bạn có thể tải file /content/nllb_pruned_novel_finetuned.zip về máy để convert sang ONNX INT8.")